In [1]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
from collections import Counter
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

dir = "/Users/julianguyen/Documents/multimodal-pmlb"
os.chdir(dir)

np.random.seed(101)

In [2]:
# load in files
atc = pd.read_csv("data/procdata/files/atac_gene.csv", index_col=0).T
rna = pd.read_csv("data/procdata/files/rna_gene.csv", index_col=0).T
cnv = pd.read_csv("data/procdata/files/cnv.csv", index_col=0).T
mut = pd.read_csv("data/procdata/files/mut.csv", index_col=0).T

# impute and log transform gene counts
imputer = SimpleImputer(strategy='median')
rna_imputed = pd.DataFrame(imputer.fit_transform(rna), index=rna.index, columns=rna.columns)
rna = np.log1p(rna_imputed.clip(lower=0))

# get targest
targets = pd.read_csv("data/procdata/files/meta.csv").set_index("PMLB_organoidID").loc[:, ["doubling_rate", "primary_tumor_site"]]
dt = targets["doubling_rate"]

# get indices
assert atc.index.equals(rna.index)
assert atc.index.equals(cnv.index)
assert atc.index.equals(mut.index)
assert atc.index.equals(targets.index)

In [3]:
# number of genes before
print("Number of genes before subset")
print("No. genes in ATAC:", len(atc.columns))
print("No. genes in RNA:", len(rna.columns))
print("No. genes in CNV:", len(cnv.columns))
print("No. genes in MUT:", len(mut.columns))

# keep gene features in >1 omics dataframe
all_genes = list(rna.columns) + list(atc.columns) + list(cnv.columns) + list(mut.columns)
gene_counts = Counter(all_genes)
keep_genes = [gene for gene, count in gene_counts.items() if count >= 2]
print("Genes in >1 omics:", len(keep_genes))

common_genes = (
    set(rna.columns)
    & set(atc.columns)
    & set(cnv.columns)
    & set(mut.columns)
)
print("Common genes across all omics:", len(common_genes))

# subset dataframes
atc = atc[[col for col in atc.columns if col in keep_genes]]
rna = rna[[col for col in rna.columns if col in keep_genes]]
cnv = cnv[[col for col in cnv.columns if col in keep_genes]]
mut = mut[[col for col in mut.columns if col in keep_genes]]

print("\nNumber of genes after subset1")
print("No. genes in ATAC:", len(atc.columns))
print("No. genes in RNA:", len(rna.columns))
print("No. genes in CNV:", len(cnv.columns))
print("No. genes in MUT:", len(mut.columns))

atc.to_csv("data/procdata/model_cg1/atc.csv", index=True)
rna.to_csv("data/procdata/model_cg1/rna.csv", index=True)
cnv.to_csv("data/procdata/model_cg1/cnv.csv", index=True)
mut.to_csv("data/procdata/model_cg1/mut.csv", index=True)
targets.to_csv("data/procdata/model_cg1/meta.csv", index=True)

# subset dataframes
atc = atc[[col for col in atc.columns if col in common_genes]]
rna = rna[[col for col in rna.columns if col in common_genes]]
cnv = cnv[[col for col in cnv.columns if col in common_genes]]
mut = mut[[col for col in mut.columns if col in common_genes]]

print("\nNumber of genes after subset2")
print("No. genes in ATAC:", len(atc.columns))
print("No. genes in RNA:", len(rna.columns))
print("No. genes in CNV:", len(cnv.columns))
print("No. genes in MUT:", len(mut.columns))

atc.to_csv("data/procdata/model_cg2/atc.csv", index=True)
rna.to_csv("data/procdata/model_cg2/rna.csv", index=True)
cnv.to_csv("data/procdata/model_cg2/cnv.csv", index=True)
mut.to_csv("data/procdata/model_cg2/mut.csv", index=True)
targets.to_csv("data/procdata/model_cg2/meta.csv", index=True)

Number of genes before subset
No. genes in ATAC: 26658
No. genes in RNA: 25247
No. genes in CNV: 25128
No. genes in MUT: 12316
Genes in >1 omics: 23477
Common genes across all omics: 9031

Number of genes after subset1
No. genes in ATAC: 19763
No. genes in RNA: 20132
No. genes in CNV: 20371
No. genes in MUT: 12088

Number of genes after subset2
No. genes in ATAC: 9031
No. genes in RNA: 9031
No. genes in CNV: 9031
No. genes in MUT: 9031
